# Mapa de Acesso à Saúde Pública em Campinas

**Projeto Integrador, Bancos de Dados Não Relacionais, PUC-Campinas, 2026**

Autores: Arthur Costa Marques, Conrado Marques, Fábio Barbosa Gonsalez, Patrick Rovaron Franco. Docente: Joice Marin.

Este notebook lê os dados do projeto, mostra os indicadores por distrito e gera o painel. Lê os JSONs de `data/`, sem internet e sem credenciais. Se `MONGO_URI` estiver definida, tenta ler do MongoDB e volta aos arquivos locais se ele falhar. O motivo dos números terem mudado em relação à versão entregue está no README, na seção "Versão 2".

## 1. Preparação

In [1]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
import dashboard
import loader

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

# No Colab, para usar o secret MONGO_URI, descomente:
# import os
# from google.colab import userdata
# os.environ["MONGO_URI"] = userdata.get("MONGO_URI")

print("Fonte de dados:", loader.fonte_em_uso())

Fonte de dados: arquivos locais (data/*.json)


## 2. Dados

`loader.carregar_documentos()` traz os documentos; `montar_df_estab` e `montar_df_indic` os transformam em DataFrames:

- `df_estab`: um estabelecimento por linha (UBS, hospital ou UPA);
- `df_indic`: um distrito de saúde por linha, com a população oficial e os indicadores.

In [2]:
docs = loader.carregar_documentos()
df_estab = loader.montar_df_estab(docs["estabelecimentos"])
df_indic = loader.montar_df_indic(docs["regioes"])

print(f"Estabelecimentos: {len(df_estab)}")
print(f"Distritos:        {len(df_indic)}")
print(f"Relações:         {len(docs['relacoes'])}")
print(f"CNES coletado em: {docs['metadados']['cnes_coletado_em']}")
df_estab[["codigo_cnes", "nome_fantasia", "categoria", "regiao", "regiao_origem", "vinculo_sus"]].head()

Estabelecimentos: 99
Distritos:        6
Relações:         99
CNES coletado em: 2026-11-05


,codigo_cnes,nome_fantasia,categoria,regiao,regiao_origem,vinculo_sus
0,411191,CENTRO DE SAUDE JARDIM BASSOLI,UBS,NOROESTE,tabela oficial de UBS,SIM
1,2022613,CENTRO DE SAUDE CAMPOS ELISEOS,UBS,SUDOESTE,tabela oficial de UBS,SIM
2,2022621,HOSPITAL E MATERNIDADE DE CAMPINAS,Hospital,LESTE,serviços do distrito no plano municipal,SIM
3,2022648,HOSPITAL IRMAOS PENTEADO E SANTA CASA DE CAMPINAS,Hospital,LESTE,serviços do distrito no plano municipal,SIM
4,2022656,CENTRO DE SAUDE SAO BERNARDO,UBS,SULESTE,tabela oficial de UBS,SIM


## 3. Indicadores por distrito

A população vem do Plano Municipal de Saúde 2026-2029, com base no Censo 2022. O parâmetro nacional é de 2.000 a 3.500 pessoas por equipe (PNAB, Portaria 2.436/2017). A pressão é igual a 100 quando há 3.500 cadastrados por equipe.

In [3]:
df_indic[["regiao", "populacao", "UBS", "equipes", "Hospital", "UPA", "hab_por_ubs", "hab_por_equipe", "cadastrados_por_equipe", "pressao", "nivel"]]

,regiao,populacao,UBS,equipes,Hospital,UPA,hab_por_ubs,hab_por_equipe,cadastrados_por_equipe,pressao,nivel
0,SUDOESTE,183424,12,47,1,0,15285,3903,4138,118,Sobrecarga
1,SUL,184941,11,35,0,1,16813,5284,4088,117,Sobrecarga
2,NORTE,197596,12,41,3,1,16466,4819,3754,107,Atenção
3,SULESTE,185853,9,35,1,1,20650,5310,3742,107,Atenção
4,LESTE,213642,9,36,5,0,23738,5934,3520,101,Atenção
5,NOROESTE,173591,16,46,1,1,10849,3774,3492,100,Dentro do parâmetro


In [4]:
# A soma dos distritos precisa fechar com o total do Censo 2022.
print("Soma dos distritos:", df_indic["populacao"].sum())
print("Total do município:", docs["metadados"]["populacao_municipio"])
assert df_indic["populacao"].sum() == docs["metadados"]["populacao_municipio"]

Soma dos distritos: 1139047
Total do município: 1139047


In [5]:
# Composição dos estabelecimentos
df_estab["sus"] = df_estab["vinculo_sus"].map({"SIM": "SUS", "NAO": "sem SUS"})
print(pd.crosstab(df_estab["categoria"], df_estab["sus"], margins=True))
print()
print("Origem do distrito de cada unidade:")
print(df_estab["regiao_origem"].value_counts().to_string())
print()
print("Critério da UBS mais próxima nos casos conhecidos:", docs["metadados"]["criterio_ubs_mais_proxima"])

sus        SUS  sem SUS  All
categoria                   
Hospital    11       15   26
UBS         69        0   69
UPA          4        0    4
All         84       15   99

Origem do distrito de cada unidade:
regiao_origem
tabela oficial de UBS                      69
UBS mais próxima                           20
serviços do distrito no plano municipal    10

Critério da UBS mais próxima nos casos conhecidos: {'acertos': 10, 'total': 10}


## 4. Painel

`dashboard.gerar()` grava o painel em um único HTML, que funciona sem internet. Para não alterar o arquivo publicado (`docs/index.html`), este notebook grava em `build/`.

In [6]:
saida = dashboard.gerar(RAIZ / "build" / "painel_notebook.html")
print(f"Painel gerado: {saida.relative_to(RAIZ)} ({saida.stat().st_size / 1_000_000:.1f} MB)")

publicado = (RAIZ / "docs" / "index.html").read_text(encoding="utf-8")
print("Igual ao docs/index.html:", saida.read_text(encoding="utf-8") == publicado)

Painel gerado: build\painel_notebook.html (1.5 MB)


Igual ao docs/index.html: True
